## Investigating The Performance of the S&P500

In [9]:
import pandas as pd

In [102]:
sp500 = pd.read_csv('../data/indices/yahoo/gspc.csv', parse_dates=True, index_col=0).sort_index(ascending=True)
sp500

,Open,High,Low,Close (split adjusted),Adj Close(dividends and splits),Volume
Date,,,,,,
2017-01-03,2251.57,2263.88,2245.13,2257.83,2257.83,3770530000
2017-01-04,2261.60,2272.82,2261.60,2270.75,2270.75,3764890000
2017-01-05,2268.18,2271.50,2260.45,2269.00,2269.00,3761820000
2017-01-06,2271.14,2282.10,2264.06,2276.98,2276.98,3339890000
2017-01-09,2273.59,2275.49,2268.90,2268.90,2268.90,3217610000
...,...,...,...,...,...,...
2021-04-12,4124.71,4131.76,4114.82,4127.99,4127.99,3578500000
2021-04-13,4130.10,4148.00,4124.43,4141.59,4141.59,3728440000
2021-04-14,4141.58,4151.69,4120.87,4124.66,4124.66,3976540000


# Goals

I need to have data that is useful when analysing the news sentiment. It would be productive to offer a module that would provide data on:

* Returns 
    * Look at the % between adj close of two days
    * Difference from adj close to the open price
    
* Market Action
    * Volatility
    * Volume
    * Trading range
    
* Drawdowns
    * Look at investment_management/1/103
    
* Other statistical measures
    * https://docs.scipy.org/doc/scipy/reference/stats.html (or the later courses in investment management)
    

Through doing this I should also think about how (if) I take the effect of a weekend into account. Can take a look at if returns on Monday are different enough.

### Returns

In [134]:
returns = sp500[['Open', 'Adj Close(dividends and splits)']]
returns['Change'] = sp500[['Adj Close(dividends and splits)']].pct_change()
returns

<ipython-input-134-aaf3d9d00e18>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns['Change'] = sp500[['Adj Close(dividends and splits)']].pct_change()


,Open,Adj Close(dividends and splits),Change
Date,,,
2017-01-03,2251.57,2257.83,NaN
2017-01-04,2261.60,2270.75,0.005722
2017-01-05,2268.18,2269.00,-0.000771
2017-01-06,2271.14,2276.98,0.003517
2017-01-09,2273.59,2268.90,-0.003549
...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196
2021-04-13,4130.10,4141.59,0.003295
2021-04-14,4141.58,4124.66,-0.004088


In [135]:
# Not sure how to do this nicely in pandas, pretty hacky
import numpy as np

close_to_open = []
close_to_open.append(np.nan)

for i in range (1, len(returns)):
    previous_close = returns.iloc[int(i) - 1]['Adj Close(dividends and splits)']
    open_price = returns.iloc[i]['Open']
    close_to_open.append((open_price-previous_close)/previous_close)

In [136]:
returns['Close to Open Change'] = close_to_open

<ipython-input-136-38268947712f>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns['Close to Open Change'] = close_to_open


In [137]:
returns

,Open,Adj Close(dividends and splits),Change,Close to Open Change
Date,,,,
2017-01-03,2251.57,2257.83,NaN,NaN
2017-01-04,2261.60,2270.75,0.005722,0.001670
2017-01-05,2268.18,2269.00,-0.000771,-0.001132
2017-01-06,2271.14,2276.98,0.003517,0.000943
2017-01-09,2273.59,2268.90,-0.003549,-0.001489
...,...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196,-0.000991
2021-04-13,4130.10,4141.59,0.003295,0.000511
2021-04-14,4141.58,4124.66,-0.004088,-0.000002


In [138]:
returns = returns[1:]
returns

,Open,Adj Close(dividends and splits),Change,Close to Open Change
Date,,,,
2017-01-04,2261.60,2270.75,0.005722,0.001670
2017-01-05,2268.18,2269.00,-0.000771,-0.001132
2017-01-06,2271.14,2276.98,0.003517,0.000943
2017-01-09,2273.59,2268.90,-0.003549,-0.001489
2017-01-10,2269.72,2268.90,0.000000,0.000361
...,...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196,-0.000991
2021-04-13,4130.10,4141.59,0.003295,0.000511
2021-04-14,4141.58,4124.66,-0.004088,-0.000002


In [139]:
returns['Formatted Change'] = 1 + returns['Change']

<ipython-input-139-d2f0dab5b958>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns['Formatted Change'] = 1 + returns['Change']


In [140]:
returns.to_csv('../data/calculations/returns.csv')

### Market Action

In [141]:
sp500.columns

Index(['Open', 'High', 'Low', 'Close (split adjusted)',
       'Adj Close(dividends and splits)', 'Volume'],
      dtype='object')

#### Volatility
I was thinking about taking the volatility of the entire period by I did'nt want the look-ahead error.

In [142]:
rolling_30_avg = sp500['Adj Close(dividends and splits)'].rolling(30).mean()
rolling_30_avg

Date
2017-01-03            NaN
2017-01-04            NaN
2017-01-05            NaN
2017-01-06            NaN
2017-01-09            NaN
                 ...     
2021-04-12    3951.577333
2021-04-13    3959.569667
2021-04-14    3968.048667
2021-04-15    3979.738667
2021-04-16    3993.638667
Name: Adj Close(dividends and splits), Length: 1079, dtype: float64

In [143]:
market_action = sp500[['Volume','Adj Close(dividends and splits)']]
market_action['30D avg price'] = rolling_30_avg
market_action

<ipython-input-143-e6cdfa0e68cb>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  market_action['30D avg price'] = rolling_30_avg


,Volume,Adj Close(dividends and splits),30D avg price
Date,,,
2017-01-03,3770530000,2257.83,NaN
2017-01-04,3764890000,2270.75,NaN
2017-01-05,3761820000,2269.00,NaN
2017-01-06,3339890000,2276.98,NaN
2017-01-09,3217610000,2268.90,NaN
...,...,...,...
2021-04-12,3578500000,4127.99,3951.577333
2021-04-13,3728440000,4141.59,3959.569667
2021-04-14,3976540000,4124.66,3968.048667


In [144]:
market_action['30D annualised volatility'] = market_action['Adj Close(dividends and splits)'].rolling(30).std() * (252 ** 0.5)
# 30 Day Rolling Volatility = Standard Deviation of the last 30 percentage changes in Total Return Price * Square-root of 252

<ipython-input-144-e28fbaf09582>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  market_action['30D annualised volatility'] = market_action['Adj Close(dividends and splits)'].rolling(30).std() * (252 ** 0.5)


In [145]:
market_action['Annualised'] =  returns['Formatted Change'].rolling(30).std() * (252 ** 0.5)

<ipython-input-145-f5fcc5f8133e>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  market_action['Annualised'] =  returns['Formatted Change'].rolling(30).std() * (252 ** 0.5)


In [146]:
market_action.to_csv('../data/calculations/market_action.csv')

In [156]:
market_action.iloc[:30]

,Volume,Adj Close(dividends and splits),30D avg price,30D annualised volatility,Annualised
Date,,,,,
2017-01-03,3770530000,2257.83,NaN,NaN,NaN
2017-01-04,3764890000,2270.75,NaN,NaN,NaN
2017-01-05,3761820000,2269.00,NaN,NaN,NaN
2017-01-06,3339890000,2276.98,NaN,NaN,NaN
2017-01-09,3217610000,2268.90,NaN,NaN,NaN
2017-01-10,3638790000,2268.90,NaN,NaN,NaN
2017-01-11,3620410000,2275.32,NaN,NaN,NaN
2017-01-12,3462130000,2270.44,NaN,NaN,NaN
2017-01-13,3081270000,2274.64,NaN,NaN,NaN


In [150]:
returns.head(1)

,Open,Adj Close(dividends and splits),Change,Close to Open Change,Formatted Change
Date,,,,,
2017-01-04,2261.6,2270.75,0.005722,0.00167,1.005722


In [152]:
trimmed_action = market_action.loc['2017-01-04':]
trimmed_action.drop(columns='Adj Close(dividends and splits)', inplace=True)
trimmed_action.head(1)

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/pandas/core/frame.py:3990: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().drop(


,Volume,30D avg price,30D annualised volatility,Annualised
Date,,,,
2017-01-04,3764890000,NaN,NaN,NaN


In [159]:
combined_market_data = returns.join(trimmed_action)
combined_market_data.loc['2018-01-01':'2018-01-20']

,Open,Adj Close(dividends and splits),Change,Close to Open Change,Formatted Change,Volume,30D avg price,30D annualised volatility,Annualised
Date,,,,,,,,,
2018-01-02,2683.73,2695.81,0.008303,0.003785,1.008303,3367250000,2648.141667,550.239150,0.064229
2018-01-03,2697.85,2713.06,0.006399,0.000757,1.006399,3538660000,2652.615333,540.782219,0.064639
2018-01-04,2719.31,2723.99,0.004029,0.002304,1.004029,3695260000,2657.343667,536.400669,0.064973
2018-01-05,2731.33,2743.15,0.007034,0.002695,1.007034,3236620000,2662.147667,562.263194,0.065301
2018-01-08,2742.67,2747.71,0.001662,-0.000175,1.001662,3242650000,2667.168667,579.994598,0.064852
2018-01-09,2751.15,2751.29,0.001303,0.001252,1.001303,3453480000,2672.131000,595.848436,0.064872
2018-01-10,2745.55,2748.23,-0.001112,-0.002086,0.998888,3576350000,2677.024667,596.377392,0.065125
2018-01-11,2752.97,2767.56,0.007034,0.001725,1.007034,3641320000,2681.708667,632.029839,0.062578
2018-01-12,2770.18,2786.24,0.006750,0.000947,1.006750,3573970000,2687.047667,678.292467,0.063875


In [161]:
combined_market_data.to_csv('../data/calculations/combined_market_data.csv')